# Lab 26 · Publish embedding models for search and RAG

Publish `text-embedding-3-small` through the gateway behind its own alias, turn sentences into vectors with the
OpenAI Python SDK, build a mini RAG app that embeds, retrieves, and answers with `gpt-5-mini` on one gateway URL and
one key, then watch a token limit stop an ingestion burst. This notebook is the Python track of the
[lab guide](README.md): it deploys the same [infra/embeddings-deployment.bicep](infra/embeddings-deployment.bicep) and
[infra/publish-embeddings.bicep](infra/publish-embeddings.bicep) as the Bicep + azd track, and runs the same code as
[infra/embed.py](infra/embed.py), [infra/rag.py](infra/rag.py), and [infra/ingest.py](infra/ingest.py).

**Check the chat model → Deploy the embedding model → Publish it → Embed → Mini RAG → Wrong API → Burst → Chart the burst → Inspect 429 → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Foundry deployment `text-embedding-3-small` | The embedding model, version 1, next to `gpt-5-mini` in the Foundry account from notebook 02. The notebook reuses it when it exists. |
| Backend model `foundry-models/text-embedding-3-small` | Registers the deployment under the provider from notebook 02 with one supported endpoint, `/openai/v1/embeddings`, and a `tokenLimit` policy: 2,000 tokens per minute for each caller identity. |
| Alias `text-embedding-3-small` | The name that clients send in the request `model` field to get vectors. |
| Alias `gpt-5-mini` | The chat alias from notebook 02, which answers the RAG questions. |
| Runtime key `app-first-call` | One key for both aliases. |
| [infra/embeddings-deployment.bicep](infra/embeddings-deployment.bicep) | Deploys the embedding model in the Foundry account, when it's missing. |
| [infra/publish-embeddings.bicep](infra/publish-embeddings.bicep) | Declares the backend model, its token limit, and the alias. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![One gateway URL and one key for both models. Each alias accepts only its own API, and the token limit caps ingestion per caller](../assets/diagrams/embeddings.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource groups of the gateway and the
  Foundry account).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-5-mini` model and its managed identity can call the Foundry account.
- Unless the Foundry account already has a `text-embedding-3-small` deployment, 20,000 tokens per minute of Global
  Standard quota for `text-embedding-3-small` in the account's region.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests openai matplotlib

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The embedding model joins the
`foundry-models` provider that notebook 02 created, next to `gpt-5-mini`.

In [ ]:
import sys
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("26-embeddings")
provider   = "foundry-models"           # model provider from notebook 02
chat       = "gpt-5-mini"               # chat model and alias from notebook 02
embeddings = "text-embedding-3-small"   # Foundry deployment, backend model, and alias name
allowance  = 2000                       # tokens per minute for each caller identity on the embedding model

### 1. Check the chat model and get a key

The cell confirms that notebook 02 published `gpt-5-mini`, then reuses the `app-first-call` key, or creates it. The
same key calls the chat model and the embedding model.

In [ ]:
lab.require(f"{lab.workspace_id}/modelProviders/{provider}/models/{chat}", "Run 02-first-model-call.ipynb first.")
display(brand.note(f"Model {provider}/{chat} exists", "ok"))

api_key, created_key = lab.api_key("app-first-call")
display(brand.key("app-first-call", api_key, created_key))

### 2. Deploy the embedding model in Foundry

The gateway publishes deployments that already exist in Foundry. When the account from notebook 02 has no
`text-embedding-3-small` deployment, [infra/embeddings-deployment.bicep](infra/embeddings-deployment.bicep) adds one
with 20,000 tokens per minute of Global Standard capacity, the same template that the Bicep + azd track runs. Clean
up deletes the deployment only if this cell created it.

In [ ]:
deployment_id      = f"{lab.foundry_id}/deployments/{embeddings}"
deployment         = lab.arm("GET", deployment_id, api_version=utils.COGNITIVE_API)
created_deployment = deployment is None
if created_deployment:
    lab.deploy("infra/embeddings-deployment.bicep", resource_group=lab.foundry_resource_group,
               foundryName=lab.foundry_account, modelName=embeddings, modelVersion="1", capacity=20)
    deployment = lab.arm("GET", deployment_id, api_version=utils.COGNITIVE_API)

deployed = deployment["properties"]["model"]
display(brand.info(f"Foundry deployment {embeddings}", [
    ("Foundry account", brand.mono(lab.foundry_account)),
    ("Model",           brand.mono(f"{deployed['name']} {deployed['version']}")),
    ("Capacity",        f"{deployment['sku']['capacity']}K tokens per minute, {deployment['sku']['name']}"),
], badge=brand.chip("Created", "ok") if created_deployment else brand.chip("Reusing", "brand")))
assert deployed["name"] == embeddings and deployment["properties"]["provisioningState"] == "Succeeded"

### 3. Publish the embedding model

[infra/publish-embeddings.bicep](infra/publish-embeddings.bicep) registers the deployment under the `foundry-models`
provider and publishes the alias that clients call. Two properties of the backend model matter here:

- `supportedEndpoints` is `["/openai/v1/embeddings"]`. The `apiFormat` values describe chat APIs only, so an
  embedding model lists its endpoint instead, and the gateway refuses every other API for it.
- `policies` holds a `tokenLimit` of 2,000 tokens per minute, counted per `identity`, so every runtime key or
  Microsoft Entra identity gets its own counter. The limit sits on the model, so every alias that routes to the
  model shares it.

The gateway's managed identity already has the Foundry User role on the account from notebook 02, so the gateway can
call the new deployment without another role assignment.

In [ ]:
published = lab.deploy("infra/publish-embeddings.bicep", gatewayName=lab.gateway_name, providerName=provider,
                       deploymentId=deployment_id, modelVersion="1", tokensPerMinute=allowance)
alias = published["aliasName"]

model = lab.arm("GET", f"{lab.workspace_id}/modelProviders/{provider}/models/{published['modelName']}")["properties"]
display(brand.info(f"Backend model {provider}/{published['modelName']}", [
    ("Supported endpoints", brand.chips(*(brand.chip(e, "violet", code=True) for e in model["supportedEndpoints"]))),
    ("Deployment",          brand.mono(f"{lab.foundry_account}/{model['deployment']['modelName']}")),
    ("Alias",               brand.mono(alias)),
    ("Embeddings URL",      brand.mono(published["embeddingsUrl"])),
]))
display(brand.policies(published["modelName"], model["policies"]))

limit = next(p for p in model["policies"] if p["type"] == "tokenLimit")
assert model["supportedEndpoints"] == ["/openai/v1/embeddings"]
assert limit["count"] == allowance and limit["period"] == "minute" and limit["counterKey"] == ["identity"]

### 4. Generate embeddings

The gateway serves embedding models at the same OpenAI-compatible base URL as chat models,
`<gateway>/default/models/openai/v1`, so the OpenAI SDK calls `client.embeddings.create` with the alias in `model`.
One call embeds a batch: the response holds one vector per input, in the same order, and the tokens that the call
used. Sentences with similar meanings get vectors that point the same way, so their cosine similarity is higher.

`text-embedding-3` models can also return shorter vectors with `dimensions`, which take less space in a vector
index. A new alias reaches the runtime within seconds, so the cell first waits until it answers.

In [ ]:
import math
from openai import OpenAI

base_url = f"{lab.gateway_url}/default/models/openai/v1"
client   = OpenAI(base_url=base_url, api_key=api_key)

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

sentences = ["How do I reset my password?",
             "I forgot the credentials I use to sign in.",
             "What's on the cafeteria menu today?"]

result  = utils.retry(lambda: client.embeddings.create(model=alias, input=sentences), statuses=(401, 404), wait=10)
vectors = [item.embedding for item in result.data]
short   = client.embeddings.create(model=alias, input=sentences[0], dimensions=256)

display(brand.info("Embeddings response", [
    ("Model",           brand.mono(result.model)),
    ("Vectors",         f"{len(vectors)}, one per sentence, with {len(vectors[0]):,} numbers each"),
    ("First numbers",   brand.mono(", ".join(f"{x:.4f}" for x in vectors[0][:3]) + ", …")),
    ("Tokens used",     f"{result.usage.total_tokens}"),
    ("dimensions=256",  f"{len(short.data[0].embedding)} numbers for the first sentence, {short.usage.total_tokens} tokens"),
], sub=f"POST {base_url}/embeddings"))
display(brand.table("Cosine similarity with the first sentence", ["Sentence", "Similarity"],
                    [(s, round(cosine(vectors[0], v), 3)) for s, v in zip(sentences[1:], vectors[1:])],
                    sub=sentences[0]))

assert len(vectors) == 3 and len(vectors[0]) == 1536 and len(short.data[0].embedding) == 256
assert cosine(vectors[0], vectors[1]) > cosine(vectors[0], vectors[2])

### 5. Build a mini RAG app

A RAG app embeds its documents once, embeds each question with the same model, ranks the documents by cosine
similarity, and asks a chat model to answer from the best matches. Eight help desk articles in a dictionary stand in
for a vector index such as Azure AI Search. Both models use the same `client`, so the same gateway URL and key:
`text-embedding-3-small` for the vectors and `gpt-5-mini` for the answer.

No article answers the second question, so the answer should say so instead of making something up.

In [ ]:
ARTICLES = {
    "password": "Reset a forgotten password in the self-service password portal. You need the authenticator app "
    "on your phone. A locked account unlocks by itself after 15 minutes.",
    "laptop-repair": "Report a cracked screen or other laptop damage in the IT portal under Hardware > Repair. "
    "Contoso ships a loaner laptop within two business days, and the repair takes about a week.",
    "vpn": "Connect to the corporate network with the Global Secure Access client. If the client shows "
    "Disconnected, sign out of Windows and sign in again.",
    "new-hire": "New employees get their laptop on their first day. Their manager requests it in the IT portal at "
    "least five business days before the start date.",
    "printers": "Add a printer from Settings > Printers. Floor printers are named after the building and the floor, "
    "for example RED-3.",
    "new-phone": "Before you wipe an old phone, register the authenticator app on the new phone from the security "
    "info page.",
    "software": "Install approved software from Company Portal. Software that isn't listed needs a request with a "
    "business justification.",
    "expenses": "Submit travel expenses within 30 days in the finance portal, with a receipt for every item over "
    "25 USD.",
}

# 1. Index: embed every article in one batch. A real app stores these vectors in a vector index.
ids     = list(ARTICLES)
indexed = client.embeddings.create(model=alias, input=[ARTICLES[i] for i in ids])
index   = {i: item.embedding for i, item in zip(ids, indexed.data)}
display(brand.note(f"Indexed {len(index)} articles with {alias}", "ok", f"{indexed.usage.total_tokens} tokens"))

def ask(question, top=2):
    """2. Retrieve the closest articles, then 3. answer from them with the chat model."""
    query   = client.embeddings.create(model=alias, input=question).data[0].embedding
    matches = sorted(((cosine(query, vector), i) for i, vector in index.items()), reverse=True)[:top]
    context = "\n".join(f"[{article}] {ARTICLES[article]}" for _, article in matches)
    answer  = client.chat.completions.create(model=chat, messages=[
        {"role": "system", "content": "Answer the question in at most two sentences, using only the articles below. "
                                      "Cite the article ID in square brackets. If the articles don't answer it, "
                                      "say so.\n\n" + context},
        {"role": "user", "content": question}])
    display(brand.table("Best matches", ["Article", "Similarity"],
                        [(brand.mono(article), round(score, 3)) for score, article in matches], sub=question))
    display(brand.reply(answer.choices[0].message.content, "Answer from the articles", model=chat,
                        tokens=(answer.usage.prompt_tokens, answer.usage.completion_tokens), sub=question))
    return matches, answer.choices[0].message.content

laptop_matches, laptop_answer = ask("My laptop screen cracked. How do I get it fixed, and how long does it take?")
dog_matches, dog_answer       = ask("Can I bring my dog to the office?")

assert laptop_matches[0][1] == "laptop-repair" and "laptop-repair" in laptop_answer
assert dog_answer

### 6. Call each alias with the wrong API

Each alias accepts only the endpoints of its backend models. A chat request to the embeddings alias, or an
embeddings request to the chat alias, gets `404` with `unsupported_endpoint`, and a model that the gateway doesn't
publish gets `404` with `model_not_found`. These calls go to the REST endpoint with `requests`, like the curl calls
in the lab guide.

The cell runs before the burst on purpose: while a caller is over the token limit, the gateway answers `429` to every
request for that alias, even one with the wrong API, because it applies the token limit before it checks the endpoint.

In [ ]:
import requests

auth = {"Authorization": f"Bearer {api_key}"}

def post(path, body, headers=auth):
    return requests.post(f"{base_url}/{path}", json=body, headers=headers, timeout=60)

def detail(response):
    """The vector size of a success, or the error code and message. A 401 or 429 body has statusCode and message."""
    body = response.json()
    if "data" in body:
        return f"{len(body['data'][0]['embedding'])} numbers, {body['usage']['total_tokens']} tokens"
    error = body.get("error") or {}
    return f"{error.get('code') or body.get('statusCode')}: {error.get('message') or body.get('message')}"

with_key           = post("embeddings", {"model": alias, "input": "Where is my order?"})
chat_to_embeddings = post("chat/completions", {"model": alias, "messages": [{"role": "user", "content": "ping"}]})
embeddings_to_chat = post("embeddings", {"model": chat, "input": "ping"})
unknown_model      = post("embeddings", {"model": "text-embedding-3-large", "input": "ping"})
without_key        = post("embeddings", {"model": alias, "input": "ping"}, headers={})

display(brand.checks("Runtime checks", [
    (f"Embeddings → {alias}",                  with_key.status_code,           200, detail(with_key)),
    (f"Chat completions → {alias}",            chat_to_embeddings.status_code, 404, detail(chat_to_embeddings)),
    (f"Embeddings → {chat}",                   embeddings_to_chat.status_code, 404, detail(embeddings_to_chat)),
    ("Embeddings → text-embedding-3-large",    unknown_model.status_code,      404, detail(unknown_model)),
    ("Embeddings without a key",               without_key.status_code,        401, detail(without_key)),
]))
assert with_key.status_code == 200
assert chat_to_embeddings.status_code == 404 and "unsupported_endpoint" in chat_to_embeddings.text
assert embeddings_to_chat.status_code == 404 and "unsupported_endpoint" in embeddings_to_chat.text
assert unknown_model.status_code == 404 and "model_not_found" in unknown_model.text
assert without_key.status_code == 401

### 7. Send an ingestion burst

An ingestion job can embed thousands of chunks in a few minutes and use up the deployment's capacity that the RAG app
needs for its questions. The token limit from step 3 gives each caller 2,000 tokens per minute. Send eight batches of
20 short chunks, about 800 tokens each, like [infra/ingest.py](infra/ingest.py). The gateway checks the counter
before it forwards a batch and counts the tokens after the response, so the batch that takes the total past 2,000
still passes, and the batches after it get `429` with `Retry-After` until the minute is over. The calls in the
earlier steps count too, so the first `429` can come one batch sooner.

The client sets `max_retries=0` to show every `429`. In a real ingestion job, keep the SDK's retries: the OpenAI SDKs
wait for `Retry-After` and send the batch again. A new token limit takes up to half a minute to apply, so the cell
repeats the burst if no batch is throttled.

In [ ]:
from openai import RateLimitError

ingest = OpenAI(base_url=base_url, api_key=api_key, max_retries=0)
chunk  = ("AI Gateway publishes models from Microsoft Foundry and other providers behind one endpoint, with API keys, "
          "token limits, cost limits, and usage telemetry for every call. Chunk {n} of the product handbook.")
batch  = [chunk.format(n=n) for n in range(1, 21)]

def send():
    """Embed one batch of chunks. Return the status, response headers, body text, and tokens used."""
    try:
        raw = ingest.embeddings.with_raw_response.create(model=alias, input=batch)
        return raw.status_code, raw.headers, raw.text, raw.parse().usage.total_tokens
    except RateLimitError as error:
        return 429, error.response.headers, error.response.text, None

def burst(count=8, attempt=1):
    """Send batches back to back. A live table shows each status and the tokens used against the allowance."""
    board = brand.Board(f"Ingestion burst against {alias}", ["Batch", "Status", "Retry-After", "Tokens used"],
                        sub=f"Attempt {attempt} · 20 chunks per batch", budget=allowance,
                        budget_label="Tokens used / allowance per minute")
    results = []
    for number in range(1, count + 1):
        status, headers, body, used = send()
        board.add([number, brand.status(status), headers.get("retry-after", "-"), used or "-"], status, used)
        results.append((status, headers, body, used))
    return results

for attempt in range(1, 5):
    results   = burst(attempt=attempt)
    throttled = [r for r in results if r[0] == 429]
    if throttled:
        break
    display(brand.note("No 429 yet. The token limit may still be applying.", "warn"))
    brand.wait(20, "before the next attempt")

assert throttled, "Expected at least one 429."
assert all(r[1].get("retry-after") for r in throttled), "A throttled response should carry Retry-After."

### 8. Chart the burst

Plot the last burst. Each bar is the tokens that one batch used, and the line is the running total that the gateway
compares with the allowance. Throttled batches never reach Foundry and use no tokens, so the line stays flat after the
first `429`. The chart follows your notebook's light or dark theme, and **Show the data** lists the numbers behind it.

In [ ]:
display(brand.usage_chart("Tokens per batch against the allowance", [(r[0], r[3]) for r in results],
                          limit=allowance, limit_label="Allowance: 2,000 tokens per minute",
                          sub=f"{alias} · app-first-call"))

### 9. Inspect a throttled response

Look at the status, headers, and body of one throttled batch. `Retry-After` tells the client how many seconds to
wait. The body isn't an OpenAI error object: it has `statusCode` and `message` instead of `error.code`, so check the
HTTP status and `Retry-After` rather than parse the body. The OpenAI SDK raises `RateLimitError` either way.

In [ ]:
status, headers, body, _ = throttled[-1]
display(brand.http(status, headers, body, "A throttled batch", sub=alias))
assert status == 429 and "Token limit is exceeded" in body

### 10. Clean up resources

The cell deletes the alias and then the backend model, and the Foundry deployment only if this notebook created it.
It deletes `app-first-call` only if this notebook created it. The `gpt-5-mini` model and alias, the provider, and the
Foundry account from notebook 02 stay for the next notebooks.

To see ingestion and chat tokens per caller in **Monitoring**, configure Application Insights first (lab 13).

In [ ]:
lab.delete(f"{lab.workspace_id}/aliases/{alias}")
lab.delete(f"{lab.workspace_id}/modelProviders/{provider}/models/{published['modelName']}")
if created_deployment:
    lab.delete(deployment_id, api_version=utils.COGNITIVE_API)
if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/app-first-call")

## Troubleshooting

- **Step 2 fails with a quota error**: the subscription has no Global Standard quota left for
  `text-embedding-3-small` in the Foundry account's region. Lower `capacity`, or free quota in the Foundry portal.
- **`404` with `model_not_found` right after step 3**: a new alias takes a few seconds to reach the runtime. Step 4
  waits for it; rerun the cell if it timed out.
- **`404` with `unsupported_endpoint` in step 4 or 5**: the request's `model` is the wrong alias for the API: the
  embeddings alias takes only `client.embeddings.create`, and the chat alias only chat and responses calls.
- **`429` in step 6 instead of `404`**: the key is over the token limit from an earlier burst, and the gateway applies
  the token limit before it checks the endpoint. Wait for `Retry-After` seconds, then rerun the cell.
- **No `429` after several bursts**: check that the request `model` is the embeddings alias, and that
  `lab.arm("GET", f"{lab.workspace_id}/modelProviders/{provider}/models/{embeddings}")["properties"]["policies"]`
  lists the token limit.
- **`KeyError: 'error'` when your code parses a `429`**: the throttled body is `{"statusCode": 429, "message": ...}`,
  not an OpenAI error object. Use the status code and `Retry-After`.
- **The OpenAI SDK waits instead of failing**: it honors `Retry-After` and retries. Set `max_retries=0` to see the
  `429`.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-26-embeddings).

## Next steps

- Give every caller its own allowance in [Throttle tokens per caller with token limits](../06-token-limits/README.md)
  and its notebook [06-token-limits.ipynb](../06-token-limits/06-token-limits.ipynb).
- See ingestion and chat tokens side by side in [Monitor usage, cost, and traces](../13-monitoring/README.md).
- Spread one alias across deployments in [Route an alias across models by priority and weight](../22-model-routing/README.md).
- Docs: [Supported APIs](https://ai.gateway.azure.com/docs/models/api-formats-and-providers#supported-apis) and
  [Releases: October 28, 2026](https://ai.gateway.azure.com/docs/releases#release-2026-10-28).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/26-embeddings/](https://azure.github.io/aigateway/labs/26-embeddings/).